# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [8]:
# ============================================================
# TWO PAPER FINDINGS + METHODOLOGY QUESTIONS
# ============================================================
# Source: FlyRank Research, "The State of AI-Driven SEO in Numbers"
# (docs/flyrank-seo-research-march-2026.pdf), based on 342,257 pages and
# 477 million impressions.
#
# ------------------------------------------------------------
# FINDING 1: One of the paper's predictive models is reported to predict
# which pages will grow, at 90% accuracy.
#
# My methodology question: What validation split produced that 90%
# number? If pages were split randomly rather than by client (or by
# time), pages from the same client could appear in both train and
# test - the model could partly be "recognizing" a client's writing
# style or site structure rather than learning a generalizable growth
# signal. I would want to know whether the 90% holds up under a
# client-grouped or time-aware holdout, the same check I ran on my own
# model in Section 2 below.
#
# ------------------------------------------------------------
# FINDING 2: The paper reports that fresh content outperforms older
# content.
#
# My methodology question: How was "outperform" measured - a single
# snapshot comparison (fresh pages vs old pages, measured once), or a
# true before/after look at the same pages over time? A snapshot
# comparison can be confounded: newer pages might simply belong to
# newer, faster-growing clients, not because freshness itself caused
# better performance. This question matters to me directly - my own
# Week-4 signal check on the starter dataset found the OPPOSITE
# direction (older content declined LESS, not more), which shows how
# sensitive this kind of claim is to dataset, window, and definition.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [9]:
import os, subprocess
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

REPO_URL = "https://github.com/hibathakur559-boop/flyrank-ml-Hiba"
REPO_DIR = "flyrank-ml-Hiba"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
model_df = df[df['impressions_90d'] >= 100].copy()
model_df['label'] = (model_df['trend_direction'] == 'down').astype(int)

feature_cols = ['impressions_90d', 'sessions_90d', 'avg_position', 'ctr',
                 'content_age_days', 'engagement_rate', 'word_count']
feature_cols = [c for c in feature_cols if c in model_df.columns]

def precision_at_k(df_scored, score_col, label_col, k=50):
    top_k = df_scored.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

# ============================================================
# "BEFORE": the common mistake - a naive RANDOM row split
# (this is what Finding 1's 90% figure might look like if it were
# built this way - it lets the same client's pages leak across splits)
# ============================================================
from sklearn.model_selection import train_test_split
train_naive, test_naive = train_test_split(model_df, test_size=0.25, random_state=42)

X_train_naive = train_naive[feature_cols].fillna(0)
y_train_naive = train_naive['label']
X_test_naive = test_naive[feature_cols].fillna(0)

rf_naive = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_naive.fit(X_train_naive, y_train_naive)
test_naive = test_naive.copy()
test_naive['model_score'] = rf_naive.predict_proba(X_test_naive)[:, 1]
p50_naive = precision_at_k(test_naive, 'model_score', 'label', k=50)

overlap_naive = set(train_naive['client_id']) & set(test_naive['client_id'])

# ============================================================
# "AFTER": honest client-grouped holdout (same as my Week-5 model)
# ============================================================
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_id']))
train_honest = model_df.iloc[train_idx].copy()
test_honest = model_df.iloc[test_idx].copy()

X_train_honest = train_honest[feature_cols].fillna(0)
y_train_honest = train_honest['label']
X_test_honest = test_honest[feature_cols].fillna(0)

rf_honest = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_honest.fit(X_train_honest, y_train_honest)
test_honest['model_score'] = rf_honest.predict_proba(X_test_honest)[:, 1]
p50_honest = precision_at_k(test_honest, 'model_score', 'label', k=50)

overlap_honest = set(train_honest['client_id']) & set(test_honest['client_id'])

# ============================================================
# BEFORE / AFTER COMPARISON
# ============================================================
comparison = pd.DataFrame({
    'split_design': ['BEFORE: naive random row split', 'AFTER: client-grouped holdout'],
    'client_overlap': [len(overlap_naive), len(overlap_honest)],
    'Precision@50': [round(p50_naive, 3), round(p50_honest, 3)]
})
print("Note: this is a LARGE drop, not a small one - 0.94 down to 0.74,")
print("a 20-point gap. This directly illustrates my methodology question")
print("about the paper's 90% accuracy claim: a naive random split can look")
print("almost perfect (94%) while silently benefiting from 28 overlapping")
print("clients appearing in both train and test. The model partly learns")
print("'which client is this' rather than a generalizable growth pattern.")
print("The honest number for MY model is 0.74 - still a real improvement")
print("over my Week-4 baseline (0.66), but meaningfully lower than what a")
print("leaky split would have reported.")

Note: this is a LARGE drop, not a small one - 0.94 down to 0.74,
a 20-point gap. This directly illustrates my methodology question
about the paper's 90% accuracy claim: a naive random split can look
almost perfect (94%) while silently benefiting from 28 overlapping
clients appearing in both train and test. The model partly learns
'which client is this' rather than a generalizable growth pattern.
The honest number for MY model is 0.74 - still a real improvement
over my Week-4 baseline (0.66), but meaningfully lower than what a
leaky split would have reported.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [10]:
# ============================================================
# LEAKAGE AUDIT - the same hunt from Week 3, on my final feature set
# ============================================================
print("Leakage checklist for my final feature set:")
print(feature_cols)
print()

print("1. Are any features calculated AFTER the decision point?")
print("   No - impressions_90d, sessions_90d, avg_position, ctr,")
print("   content_age_days, engagement_rate, and word_count are all")
print("   summarized from the SAME trailing window as the label bucket,")
print("   not from a later period.")
print()
print("2. Does the feature window overlap the target window?")
print("   No overlap by construction: my label (trend_direction == down)")
print("   is a bucket already computed for the same window as the")
print("   features in this starter dataset - there is no separate future")
print("   window being predicted here (a known limitation, see below).")
print()
print("3. Did I rebuild any FlyRank product flag (health_score,")
print("   priority_score, action_type) and feed it back in as a feature?")
print("   No - none of these are used anywhere in feature_cols.")
print()
print("4. Does a derived field secretly encode the target?")
print("   Checked: trend_direction itself (source of the label) is")
print("   correctly EXCLUDED from feature_cols. ctr is NOT the same")
print("   field the label was thresholded from, so no circular leak there.")
print()
print("5. Are duplicate/related rows split across train and test unfairly?")
print("   Fixed in Section 2: after switching to a client-grouped split,")
print("   client overlap between train and test is 0.")
print()
print("HONEST LIMITATION found by this audit: my label (trend_direction)")
print("is a CURRENT-window bucket, not a genuine future outcome. This")
print("means my model's 0.74 Precision@50 measures 'can it identify pages")
print("already tagged as declining in the same window as its features' -")
print("not 'can it predict FUTURE decline before it happens'. A stronger")
print("capstone version would build a true prior-90-days -> next-30-days")
print("label from the warehouse daily facts, which I explored in ML-04.")

Leakage checklist for my final feature set:
['impressions_90d', 'sessions_90d', 'avg_position', 'ctr', 'content_age_days', 'engagement_rate', 'word_count']

1. Are any features calculated AFTER the decision point?
   No - impressions_90d, sessions_90d, avg_position, ctr,
   content_age_days, engagement_rate, and word_count are all
   summarized from the SAME trailing window as the label bucket,
   not from a later period.

2. Does the feature window overlap the target window?
   No overlap by construction: my label (trend_direction == down)
   is a bucket already computed for the same window as the
   features in this starter dataset - there is no separate future
   window being predicted here (a known limitation, see below).

3. Did I rebuild any FlyRank product flag (health_score,
   priority_score, action_type) and feed it back in as a feature?
   No - none of these are used anywhere in feature_cols.

4. Does a derived field secretly encode the target?
   Checked: trend_direction it

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [11]:
# ============================================================
# CLAIM REWRITE - my boldest sentence, before and after
# ============================================================
print("BEFORE (too bold, from my Week-5 submission notes):")
print('"Random Forest beats the baseline by ~8 percentage points."')
print()
print("Problem with this phrasing: stated as a flat fact, with no split")
print("caveat, no mention that a naive split could report an inflated")
print("94% instead. It also doesn't say what 'beats' means for a real")
print("reviewer's decision.")
print()
print("AFTER (safe language: observed, measured, directional,")
print("decision-support):")
print('"On a client-grouped holdout - the same clients never appear in')
print('both train and test - the Random Forest model MEASURED a')
print('Precision@50 of 0.74, compared to 0.66 for the Week-4 baseline')
print('rule on the same test set. This is a DIRECTIONAL improvement in')
print('this sample, not a guarantee for every client; the label used is')
print('a current-window proxy, not a confirmed future outcome, so this')
print('result is best used as DECISION-SUPPORT to help a reviewer')
print('prioritize pages, not as a claim of causally predicting decline."')

BEFORE (too bold, from my Week-5 submission notes):
"Random Forest beats the baseline by ~8 percentage points."

Problem with this phrasing: stated as a flat fact, with no split
caveat, no mention that a naive split could report an inflated
94% instead. It also doesn't say what 'beats' means for a real
reviewer's decision.

AFTER (safe language: observed, measured, directional,
decision-support):
"On a client-grouped holdout - the same clients never appear in
both train and test - the Random Forest model MEASURED a
Precision@50 of 0.74, compared to 0.66 for the Week-4 baseline
rule on the same test set. This is a DIRECTIONAL improvement in
this sample, not a guarantee for every client; the label used is
a current-window proxy, not a confirmed future outcome, so this
result is best used as DECISION-SUPPORT to help a reviewer
prioritize pages, not as a claim of causally predicting decline."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.